# Notebook 3: Path-Dependent Options
**Project:** Monte Carlo Risk & Derivatives Pricing
**Author:** Niraj Neupane | github.com/nirajneupane17

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations')


In [ ]:
from options_pricing import bs_call, mc_asian, mc_barrier_ko, mc_lookback
S0,K,r,T,sigma,B = 100,100,0.05,1.0,0.20,120
euro  = bs_call(S0,K,T,r,sigma)
asian = mc_asian(S0,K,T,r,sigma)
barr  = mc_barrier_ko(S0,K,B,T,r,sigma)
look  = mc_lookback(S0,T,r,sigma)
print(f'European Call  : ${euro:.4f}')
print(f'Asian Call     : ${asian:.4f} ({(asian/euro-1)*100:.1f}% vs European)')
print(f'Barrier KO     : ${barr:.4f} ({(barr/euro-1)*100:.1f}% vs European)')
print(f'Lookback Call  : ${look:.4f} ({(look/euro-1)*100:.1f}% vs European)')

In [ ]:
option_types=["European
Call","Asian
Call","Barrier
Knock-Out","Lookback
Call"]
prices=[euro,asian,barr,look]
colors_o=["#185FA5","#E24B4A","#1D9E75","#7B1FA2"]
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
bars=ax1.bar(option_types,prices,color=colors_o,alpha=0.85,edgecolor="white",width=0.5)
for bar,v in zip(bars,prices):
    ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,f"${v:.2f}",ha="center",va="bottom",fontsize=10,fontweight="bold")
ax1.set_title(f'Option Prices: European vs Exotic (S={S0}, K={K}, sigma={int(sigma*100)}%)')
ax1.set_ylabel('Option Price ($)')
barriers=np.arange(105,145,5)
barr_p=[mc_barrier_ko(S0,K,b,T,r,sigma,n_sims=20000) for b in barriers]
ax2=axes[1]
ax2.plot(barriers,barr_p,'o-',color='#185FA5',linewidth=2,markersize=7,label='KO Barrier price')
ax2.axhline(euro,color='#E24B4A',linewidth=1.5,linestyle='--',label=f'European Call=${euro:.2f}')
ax2.fill_between(barriers,barr_p,euro,alpha=0.15,color='#E24B4A')
ax2.set_title('Knock-Out Barrier Option vs Barrier Level')
ax2.set_xlabel('Barrier Level (B)'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/04_path_dependent_options.png',dpi=150,bbox_inches='tight')
plt.show()